Name: Liza Jivnani
UID: U16181370

# Assignment Objective:

Modify the model’s text preprocessing by changing from character-level tokenization to word-level tokenization. Compare the performance of both tokenization methods. Additionally, perform hyper-parameter optimization by experimenting with various settings (learning rate, hidden layers, hidden sizes, batch sizes, optimizers, and activation functions) and report your findings.

# Part 1: Character-level Tokenization Vs. Word-level Tokenization

## Imports and Random Seeds


In [34]:

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score

tf.random.set_seed(123)
np.random.seed(123)


import tensorflow as tf
print(tf.test.gpu_device_name())
print('GPU:', tf.config.list_physical_devices(device_type='GPU'))
print(tf.test.is_gpu_available())


/device:GPU:0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU')]
True


I0000 00:00:1740022497.825204  280541 gpu_device.cc:2022] Created device /device:GPU:0 with 22991 MB memory:  -> device: 0, name: NVIDIA TITAN RTX, pci bus id: 0000:3b:00.0, compute capability: 7.5
I0000 00:00:1740022497.825985  280541 gpu_device.cc:2022] Created device /device:GPU:1 with 22991 MB memory:  -> device: 1, name: NVIDIA TITAN RTX, pci bus id: 0000:86:00.0, compute capability: 7.5
I0000 00:00:1740022497.826453  280541 gpu_device.cc:2022] Created device /device:GPU:2 with 10699 MB memory:  -> device: 2, name: NVIDIA TITAN Xp, pci bus id: 0000:af:00.0, compute capability: 6.1
I0000 00:00:1740022497.834949  280541 gpu_device.cc:2022] Created device /device:GPU:0 with 22991 MB memory:  -> device: 0, name: NVIDIA TITAN RTX, pci bus id: 0000:3b:00.0, compute capability: 7.5
I0000 00:00:1740022497.836407  280541 gpu_device.cc:2022] Created device /device:GPU:1 with 22991 MB memory:  -> device: 1, name: NVIDIA TITAN RTX, pci bus id: 0000:86:00.0, compute capability: 7.5
I0000 00:00

## Base MLP

In [35]:
# -------------------------------
# Original MLP Class Definition
# -------------------------------
class MLP(object):
    def __init__(self, size_input, size_hidden1, size_hidden2, size_hidden3, size_output, device=None):
        """
        size_input: int, size of input layer
        size_hidden1: int, size of the 1st hidden layer
        size_hidden2: int, size of the 2nd hidden layer
        size_hidden3: int, size of the 3rd hidden layer (not used in compute_output here)
        size_output: int, size of output layer
        device: str or None, either 'cpu' or 'gpu' or None.
        """
        self.size_input = size_input
        self.size_hidden1 = size_hidden1
        self.size_hidden2 = size_hidden2
        self.size_hidden3 = size_hidden3  # (Currently not used in the forward pass)
        self.size_output = size_output
        self.device = device

        # Initialize weights and biases for first hidden layer
        self.W1 = tf.Variable(tf.random.normal([self.size_input, self.size_hidden1], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([1, self.size_hidden1]))

        # Initialize weights and biases for second hidden layer
        self.W2 = tf.Variable(tf.random.normal([self.size_hidden1, self.size_hidden2], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([1, self.size_hidden2]))

        # Initialize weights and biases for output layer
        self.W3 = tf.Variable(tf.random.normal([self.size_hidden2, self.size_output], stddev=0.1))
        self.b3 = tf.Variable(tf.zeros([1, self.size_output]))

        # List of variables to update during backpropagation
        self.variables = [self.W1, self.W2, self.W3, self.b1, self.b2, self.b3]

    def forward(self, X):
        """
        Forward pass.
        X: Tensor, inputs.
        """
        if self.device is not None:
            with tf.device('gpu:0' if self.device == 'gpu' else 'cpu'):
                self.y = self.compute_output(X)
        else:
            self.y = self.compute_output(X)
        return self.y

    def loss(self, y_pred, y_true):
        """
        Computes the loss between predicted and true outputs.
        y_pred: Tensor of shape (batch_size, size_output)
        y_true: Tensor of shape (batch_size, size_output)
        """
        y_true_tf = tf.cast(y_true, dtype=tf.float32)
        y_pred_tf = tf.cast(y_pred, dtype=tf.float32)
        cce = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
        loss_x = cce(y_true_tf, y_pred_tf)
        return loss_x

    def backward(self, X_train, y_train):
        """
        Backward pass: compute gradients of the loss with respect to the variables.
        """
        with tf.GradientTape() as tape:
            predicted = self.forward(X_train)
            current_loss = self.loss(predicted, y_train)
        grads = tape.gradient(current_loss, self.variables)
        return grads

    def compute_output(self, X):
        """
        Custom method to compute the output tensor during the forward pass.
        """
        # Cast X to float32
        X_tf = tf.cast(X, dtype=tf.float32)
        # First hidden layer
        h1 = tf.matmul(X_tf, self.W1) + self.b1
        z1 = tf.nn.relu(h1)
        # Second hidden layer
        h2 = tf.matmul(z1, self.W2) + self.b2
        z2 = tf.nn.relu(h2)
        # Output layer (logits)
        output = tf.matmul(z2, self.W3) + self.b3
        return output

## Defining Tokenizers

In [3]:
# -------------------------------
# Tokenizer and Preprocessing Functions
# -------------------------------

#-------------------------
# Character-Level Tokenizer
# -------------------------
def char_level_tokenizer(texts, num_words=None):
    """
    Create and fit a character-level tokenizer.

    Args:
        texts (list of str): List of texts.
        num_words (int or None): Maximum number of tokens to keep.

    Returns:
        tokenizer: A fitted Tokenizer instance.
    """
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=num_words, char_level=True, lower=True)
    tokenizer.fit_on_texts(texts)
    return tokenizer

#-------------------------
# Word-Level Tokenizer
# -------------------------
def word_level_tokenizer(texts, num_words=None):
    """
    Create and fit a character-level tokenizer.

    Args:
        texts (list of str): List of texts.
        num_words (int or None): Maximum number of tokens to keep.

    Returns:
        tokenizer: A fitted Tokenizer instance.
    """
    tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=num_words, char_level=False, lower=True)
    tokenizer.fit_on_texts(texts)
    return tokenizer

def texts_to_bow(tokenizer, texts):
    """
    Convert texts to a bag-of-characters representation.

    Args:
        tokenizer: A fitted character-level Tokenizer.
        texts (list of str): List of texts.

    Returns:
        Numpy array representing the binary bag-of-characters for each text.
    """
    # texts_to_matrix with mode 'binary' produces a fixed-length binary vector per text.
    matrix = tokenizer.texts_to_matrix(texts, mode='binary')
    return matrix

def one_hot_encode(labels, num_classes=2):
    """
    Convert numeric labels to one-hot encoded vectors.
    """
    return np.eye(num_classes)[labels]


## Load Dataset

In [37]:
from sys import exit as e
# -------------------------------
# Load and Prepare the IMDB Dataset
# -------------------------------
print("Loading IMDB dataset...")
# Load the IMDB reviews dataset with the 'as_supervised' flag so that we get (text, label) pairs.
(ds_train, ds_test), ds_info = tfds.load('imdb_reviews',
                                           split=['train', 'test'],
                                           as_supervised=True,
                                           with_info=True)

# Convert training dataset to lists.
train_texts = []
train_labels = []
for text, label in tfds.as_numpy(ds_train):
    # Decode byte strings to utf-8 strings.
    train_texts.append(text.decode('utf-8'))
    train_labels.append(label)

train_labels = np.array(train_labels)

# Create a validation set from the training data (20% for validation).
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=42)

# Convert test dataset to lists.
test_texts = []
test_labels = []
for text, label in tfds.as_numpy(ds_test):
    test_texts.append(text.decode('utf-8'))
    test_labels.append(label)
test_labels = np.array(test_labels)

print(f"Train samples: {len(train_texts)}, Validation samples: {len(val_texts)}, Test samples: {len(test_texts)}")



Loading IMDB dataset...
Train samples: 20000, Validation samples: 5000, Test samples: 25000


## Preprocessing

In [39]:
# -------------------------------
# Preprocessing: Tokenization and Vectorization
# -------------------------------
# Build the character-level tokenizer on the training texts.
tokenizer = char_level_tokenizer(train_texts)
print("Tokenizer vocabulary size:", len(tokenizer.word_index) + 1)

# Convert texts to bag-of-characters representation.
X_train = texts_to_bow(tokenizer, train_texts)
X_val   = texts_to_bow(tokenizer, val_texts)
X_test  = texts_to_bow(tokenizer, test_texts)

print(X_train)


# Convert labels to one-hot encoding.
y_train = one_hot_encode(train_labels)
y_val   = one_hot_encode(val_labels)
y_test  = one_hot_encode(test_labels)


print(y_train)




Tokenizer vocabulary size: 134
[[0. 1. 1. ... 0. 0. 0.]
 [0. 1. 1. ... 0. 0. 0.]
 [0. 1. 1. ... 0. 0. 0.]
 ...
 [0. 1. 1. ... 0. 0. 0.]
 [0. 1. 1. ... 0. 0. 0.]
 [0. 1. 1. ... 0. 0. 0.]]
[[0. 1.]
 [0. 1.]
 [0. 1.]
 ...
 [1. 0.]
 [0. 1.]
 [1. 0.]]


## Training and testing for both word-level and chracter-level tokenization

In [ ]:

# -------------------------------
# Model Setup
# -------------------------------
# The input size is determined by the dimension of the bag-of-characters vector.
size_input = X_train.shape[1]
# Set hidden layer sizes as desired.
size_hidden1 = 128
size_hidden2 = 64
size_hidden3 = 32  # Placeholder (not used in the forward pass)
size_output  = 2

# Instantiate the MLP model.
model = MLP(size_input, size_hidden1, size_hidden2, size_hidden3, size_output, device=None)

# Define the optimizer.
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# -------------------------------
# Training Parameters and Loop
# -------------------------------
batch_size = 128
epochs = 10
num_batches = int(np.ceil(X_train.shape[0] / batch_size))

print("\nStarting training...\n")
for epoch in range(epochs):
    # Shuffle training data at the start of each epoch.
    indices = np.arange(X_train.shape[0])
    np.random.shuffle(indices)
    X_train = X_train[indices]
    y_train = y_train[indices]

    epoch_loss = 0
    for i in range(num_batches):
        start = i * batch_size
        end = min((i+1) * batch_size, X_train.shape[0])
        X_batch = X_train[start:end]
        y_batch = y_train[start:end]

        # Compute gradients and update weights.
        # with tf.GradientTape() as tape:
        #     predictions = model.forward(X_batch)
        #     loss_value = model.loss(predictions, y_batch)
        # grads = tape.gradient(loss_value, model.variables)
        predictions = model.forward(X_batch)
        loss_value = model.loss(predictions, y_batch)
        grads = model.backward(X_batch, y_batch)
        optimizer.apply_gradients(zip(grads, model.variables))
        epoch_loss += loss_value.numpy() * (end - start)

    epoch_loss /= X_train.shape[0]

    # Evaluate on validation set.
    val_logits = model.forward(X_val)
    val_loss = model.loss(val_logits, y_val).numpy()
    val_preds = np.argmax(val_logits.numpy(), axis=1)
    true_val = np.argmax(y_val, axis=1)
    accuracy = np.mean(val_preds == true_val)
    precision = precision_score(true_val, val_preds)
    recall = recall_score(true_val, val_preds)

    print(f"Epoch {epoch+1:02d} | Training Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")


print("finished training")

# -------------------------------
# Final Evaluation on Test Set
# -------------------------------


# Split the test set into 10 batches
num_test_batches = 10
test_batch_size = X_test.shape[0] // num_test_batches

test_accuracies = []
test_precisions = []
test_recalls = [] and Test 1 else X_test.shape[0]
    
    X_test_batch = X_test[start:end]
    y_test_batch = y_test[start:end]
    
    test_logits = model.forward(X_test_batch)
    test_loss = model.loss(test_logits, y_test_batch).numpy()
    test_preds = np.argmax(test_logits.numpy(), axis=1)
    true_test = np.argmax(y_test_batch, axis=1)
    
    test_accuracy = np.mean(test_preds == true_test)
    test_precision = precision_score(true_test, test_preds)
    test_recall = recall_score(true_test, test_preds)
    
    test_accuracies.append(test_accuracy)
    test_precisions.append(test_precision)
    test_recalls.append(test_recall)
    test_losses.append(test_loss)

# Calculate average metrics
avg_test_accuracy = np.mean(test_accuracies)
avg_test_precision = np.mean(test_precisions)
avg_test_recall = np.mean(test_recalls)
avg_test_loss = np.mean(test_losses)

print(f"Average Test Loss: {avg_test_loss:.4f}")
print(f"Average Test Accuracy: {avg_test_accuracy:.4f}")
print(f"Average Test Precision: {avg_test_precision:.4f}")
print(f"Average Test Recall: {avg_test_recall:.4f}")

# Print individual batch results
print("\nIndividual Batch Results:")
for i, (acc, prec, rec) in enumerate(zip(test_accuracies, test_precisions, test_recalls)):
    print(f"Batch {i+1}: Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")



Starting training...

Epoch 01 | Training Loss: 0.6723 | Val Loss: 0.6679 | Accuracy: 0.5876 | Precision: 0.6236 | Recall: 0.3767
Epoch 02 | Training Loss: 0.6612 | Val Loss: 0.6614 | Accuracy: 0.6082 | Precision: 0.5813 | Recall: 0.6856
Epoch 03 | Training Loss: 0.6585 | Val Loss: 0.6603 | Accuracy: 0.6088 | Precision: 0.5818 | Recall: 0.6869
Epoch 04 | Training Loss: 0.6556 | Val Loss: 0.6595 | Accuracy: 0.6122 | Precision: 0.5944 | Recall: 0.6300
Epoch 05 | Training Loss: 0.6538 | Val Loss: 0.6597 | Accuracy: 0.6084 | Precision: 0.6149 | Recall: 0.5144
Epoch 06 | Training Loss: 0.6532 | Val Loss: 0.6605 | Accuracy: 0.6052 | Precision: 0.5873 | Recall: 0.6246
Epoch 07 | Training Loss: 0.6502 | Val Loss: 0.6604 | Accuracy: 0.6036 | Precision: 0.5730 | Recall: 0.7153
Epoch 08 | Training Loss: 0.6466 | Val Loss: 0.6583 | Accuracy: 0.6128 | Precision: 0.5883 | Recall: 0.6704
Epoch 09 | Training Loss: 0.6441 | Val Loss: 0.6608 | Accuracy: 0.6120 | Precision: 0.5874 | Recall: 0.6708
Epoch

# Part 2: Hyper parameter Optimization

## Imports and Random Seeds

In [82]:

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, accuracy_score
import itertools

tf.random.set_seed(000)
np.random.seed(000)


import tensorflow as tf
print(tf.test.gpu_device_name())
print('GPU:', tf.config.list_physical_devices(device_type='GPU'))
print(tf.test.is_gpu_available())

/device:GPU:0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU')]
True


I0000 00:00:1740034578.243038  280541 gpu_device.cc:2022] Created device /device:GPU:0 with 22991 MB memory:  -> device: 0, name: NVIDIA TITAN RTX, pci bus id: 0000:3b:00.0, compute capability: 7.5
I0000 00:00:1740034578.243441  280541 gpu_device.cc:2022] Created device /device:GPU:1 with 22991 MB memory:  -> device: 1, name: NVIDIA TITAN RTX, pci bus id: 0000:86:00.0, compute capability: 7.5
I0000 00:00:1740034578.243690  280541 gpu_device.cc:2022] Created device /device:GPU:2 with 10699 MB memory:  -> device: 2, name: NVIDIA TITAN Xp, pci bus id: 0000:af:00.0, compute capability: 6.1
I0000 00:00:1740034578.247635  280541 gpu_device.cc:2022] Created device /device:GPU:0 with 22991 MB memory:  -> device: 0, name: NVIDIA TITAN RTX, pci bus id: 0000:3b:00.0, compute capability: 7.5
I0000 00:00:1740034578.247891  280541 gpu_device.cc:2022] Created device /device:GPU:1 with 22991 MB memory:  -> device: 1, name: NVIDIA TITAN RTX, pci bus id: 0000:86:00.0, compute capability: 7.5
I0000 00:00

## Modified MLP for Grid Search

In [ ]:
class MLP(object):
    def __init__(self, size_input, size_hidden1, size_hidden2, size_hidden3, size_output,
                 activation='relu', device=None):  # Added activation
        """
        size_input: int, size of input layer
        size_hidden1: int, size of the 1st hidden layer
        size_hidden2: int, size of the 2nd hidden layer
        size_hidden3: int, size of the 3rd hidden layer (not used in compute_output here)
        size_output: int, size of output layer
        activation: str, activation function ('relu', 'tanh', 'leaky_relu')
        device: str or None, either 'cpu' or 'gpu' or None.
        """
        self.size_input = size_input
        self.size_hidden1 = size_hidden1
        self.size_hidden2 = size_hidden2
        self.size_hidden3 = size_hidden3
        self.size_output = size_output
        self.activation = activation  # Store activation function
        self.device = device

        # Initialize weights and biases for first hidden layer
        self.W1 = tf.Variable(tf.random.normal([self.size_input, self.size_hidden1], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([1, self.size_hidden1]))

        # Initialize weights and biases for second hidden layer
        self.W2 = tf.Variable(tf.random.normal([self.size_hidden1, self.size_hidden2], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([1, self.size_hidden2]))

        # Initialize weights and biases for third hidden layer (if needed)
        self.W3 = tf.Variable(tf.random.normal([self.size_hidden2, self.size_output], stddev=0.1))
        self.b3 = tf.Variable(tf.zeros([1, self.size_output]))

        # List of variables to update during backpropagation
        self.variables = [self.W1, self.W2, self.W3, self.b1, self.b2, self.b3]

    def forward(self, X):
        """
        Forward pass.
        X: Tensor, inputs.
        """
        if self.device is not None:
            with tf.device('gpu:0' if self.device == 'gpu' else 'cpu'):
                self.y = self.compute_output(X)
        else:
            self.y = self.compute_output(X)
        return self.y

    def loss(self, y_pred, y_true):
        """
        Computes the loss between predicted and true outputs.
        y_pred: Tensor of shape (batch_size, size_output)
        y_true: Tensor of shape (batch_size, size_output)
        """
        y_true_tf = tf.cast(y_true, dtype=tf.float32)
        y_pred_tf = tf.cast(y_pred, dtype=tf.float32)
        cce = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
        loss_x = cce(y_true_tf, y_pred_tf)
        return loss_x

    def backward(self, X_train, y_train):
        """
        Backward pass: compute gradients of the loss with respect to the variables.
        """
        with tf.GradientTape() as tape:
            predicted = self.forward(X_train)
            current_loss = self.loss(predicted, y_train)
        grads = tape.gradient(current_loss, self.variables)
        return grads

    def compute_output(self, X):
        """
        Custom method to compute the output tensor during the forward pass.
        """
        # Cast X to float32
        X_tf = tf.cast(X, dtype=tf.float32)
        
        # First hidden layer
        h1 = tf.matmul(X_tf, self.W1) + self.b1
        z1 = self.apply_activation(h1)

        # Second hidden layer
        h2 = tf.matmul(z1, self.W2) + self.b2
        z2 = self.apply_activation(h2)

        # Output layer (logits)
        output = tf.matmul(z2, self.W3) + self.b3
        return output

    def apply_activation(self, tensor):

        """Applies the specified activation function."""
        if self.activation == 'relu':
            return tf.nn.relu(tensor)
        elif self.activation == 'tanh':
            return tf.nn.tanh(tensor)
        else:
            raise ValueError(f"Invalid activation function: {self.activation}") and Test


## Load Dataset and Preprocess using Word-level Tokenizer

In [84]:
# -------------------------------
# Load and Prepare the IMDB Dataset
# -------------------------------
print("Loading IMDB dataset...")
# Load the IMDB reviews dataset with the 'as_supervised' flag so that we get (text, label) pairs.
(ds_train, ds_test), ds_info = tfds.load('imdb_reviews',
                                           split=['train', 'test'],
                                           as_supervised=True,
                                           with_info=True)

# Convert training dataset to lists.
train_texts = []
train_labels = []
for text, label in tfds.as_numpy(ds_train):
    # Decode byte strings to utf-8 strings.
    train_texts.append(text.decode('utf-8'))
    train_labels.append(label)
train_labels = np.array(train_labels)

# Create a validation set from the training data (20% for validation).
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=42)

# Convert test dataset to lists.
test_texts = []
test_labels = []
for text, label in tfds.as_numpy(ds_test):
    test_texts.append(text.decode('utf-8'))
    test_labels.append(label)
test_labels = np.array(test_labels)

print(f"Train samples: {len(train_texts)}, Validation samples: {len(val_texts)}, Test samples: {len(test_texts)}")


# -------------------------------
# Preprocessing: Tokenization and Vectorization
# -------------------------------

# Build the character-level tokenizer on the training texts.
tokenizer = word_level_tokenizer(train_texts)

print("Tokenizer vocabulary size:", len(tokenizer.word_index) + 1)

# Convert texts to bag-of-characters representation.
X_train = texts_to_bow(tokenizer, train_texts)
X_val   = texts_to_bow(tokenizer, val_texts)
X_test  = texts_to_bow(tokenizer, test_texts)

print(X_train)


# Convert labels to one-hot encoding.
y_train = one_hot_encode(train_labels)
y_val   = one_hot_encode(val_labels)
y_test  = one_hot_encode(test_labels)


print(y_train)


Loading IMDB dataset...
Train samples: 20000, Validation samples: 5000, Test samples: 25000
Tokenizer vocabulary size: 80169
[[0. 1. 1. ... 0. 0. 0.]
 [0. 1. 1. ... 0. 0. 0.]
 [0. 1. 1. ... 0. 0. 0.]
 ...
 [0. 1. 1. ... 0. 0. 0.]
 [0. 1. 1. ... 1. 1. 1.]
 [0. 1. 1. ... 0. 0. 0.]]
[[0. 1.]
 [0. 1.]
 [0. 1.]
 ...
 [1. 0.]
 [0. 1.]
 [1. 0.]]


## Define the Hyperparameter Grid 

In [67]:
# -------------------------------
# Hyperparameter Grid
# -------------------------------

learning_rates = [0.001, 0.0001]
hidden_layers = [2, 3]
hidden_sizes = [512,]
batch_sizes = [128, ]
optimizers_list = ['Adam', 'SGD',]
activation_functions = ['relu', 'tanh', ]

# Create a list of all hyperparameter combinations
param_grid = list(itertools.product(learning_rates, hidden_layers, hidden_sizes, batch_sizes, optimizers_list, activation_functions))


## Training and optimizing using Grid Search

In [ ]:
best_accuracy = 0.0
best_params = None
all_results = []

for params in param_grid:
    lr, num_layers, hidden_size, batch_size, optimizer_name, activation = params

    print(f"\nTraining with parameters: lr={lr}, layers={num_layers}, size={hidden_size}, "
          f"batch={batch_size}, optimizer={optimizer_name}, activation={activation}")

    # Define model architecture based on the number of hidden layers
    size_input = X_train.shape[1]
    size_output = 2


    # passing 0 neurons to eliminate the layer
    if num_layers == 2:
        model = MLP(size_input, hidden_size, 64, 0, size_output, activation=activation)
    elif num_layers == 3:
        model = MLP(size_input, hidden_size, 64, 32, size_output, activation=activation)
    else:
        raise ValueError("Invalid number of hidden layers")

    # Define the optimizer
    if optimizer_name == 'Adam':
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_name == 'SGD':
        optimizer = tf.keras.optimizers.SGD(learning_rate=lr)
    elif optimizer_name == 'RMSProp':
        optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)
    else:
        raise ValueError(f"Invalid optimizer: {optimizer_name}")

    epochs = 5  # Reduced epochs for faster grid search

    num_batches = int(np.ceil(X_train.shape[0] / batch_size))

    for epoch in range(epochs):
        # Shuffle training data at the start of each epoch.
        indices = np.arange(X_train.shape[0])
        np.random.shuffle(indices)
        X_train = X_train[indices]
        y_train = y_train[indices]

        epoch_loss = 0
        for i in range(num_batches):
            start = i * batch_size
            end = min((i+1) * batch_size, X_train.shape[0])
            X_batch = X_train[start:end]
            y_batch = y_train[start:end]

            # Compute gradients and update weights.
            predictions = model.forward(X_batch)
            loss_value = model.loss(predictions, y_batch)
            grads = model.backward(X_batch, y_batch)
            optimizer.apply_gradients(zip(grads, model.variables))
            epoch_loss += loss_value.numpy() * (end - start)

        epoch_loss /= X_train.shape[0]

        # Evaluate on validation set.
        val_logits = model.forward(X_val)
        val_loss = model.loss(val_logits, y_val).numpy()
        val_preds = np.argmax(val_logits.numpy(), axis=1)
        true_val = np.argmax(y_val, axis=1)
        accuracy = accuracy_score(true_val, val_preds)  # Using accuracy_score
        precision = precision_score(true_val, val_preds, zero_division=0) # prevent zero division
        recall = recall_score(true_val, val_preds, zero_division=0)   # prevent zero division

        print(f"Epoch {epoch+1:02d} | Training Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

    # Store results
    result = {
        'learning_rate': lr,
        'hidden_layers': num_layers,
        'hidden_size': hidden_size,
        'batch_size': batch_size,
        'optimizer': optimizer_name,
        'activation': activation,
        'val_accuracy': accuracy,
        'val_loss': val_loss,
        'precision': precision,
        'recall': recall
    }
    all_results.append(result)

    # Update best accuracy
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = params

# -------------------------------
# Identify Best Model and Hyperparameters
# -------------------------------


print("\nGrid search complete.")
print(f"Best Validation Accuracy: {best_accuracy:.4f}")
print(f"Best Hyperparameters: lr={best_params[0]}, layers={best_params[1]}, size={best_params[2]}, "
      f"batch={best_params[3]}, optimizer={best_params[4]}, activation={best_params[5]}")

 and Test


Training with parameters: lr=0.001, layers=2, size=512, batch=128, optimizer=Adam, activation=relu
Epoch 01 | Training Loss: 0.3712 | Val Loss: 0.2803 | Accuracy: 0.8846 | Precision: 0.8600 | Recall: 0.9101
Epoch 02 | Training Loss: 0.0595 | Val Loss: 0.3335 | Accuracy: 0.8778 | Precision: 0.9093 | Recall: 0.8309
Epoch 03 | Training Loss: 0.0083 | Val Loss: 0.4026 | Accuracy: 0.8906 | Precision: 0.8738 | Recall: 0.9051
Epoch 04 | Training Loss: 0.0014 | Val Loss: 0.4503 | Accuracy: 0.8900 | Precision: 0.8878 | Recall: 0.8849
Epoch 05 | Training Loss: 0.0005 | Val Loss: 0.4849 | Accuracy: 0.8904 | Precision: 0.8785 | Recall: 0.8981

Training with parameters: lr=0.001, layers=2, size=512, batch=128, optimizer=Adam, activation=tanh
Epoch 01 | Training Loss: 0.3365 | Val Loss: 0.2633 | Accuracy: 0.8888 | Precision: 0.9100 | Recall: 0.8552
Epoch 02 | Training Loss: 0.0655 | Val Loss: 0.3073 | Accuracy: 0.8846 | Precision: 0.8938 | Recall: 0.8647
Epoch 03 | Training Loss: 0.0124 | Val Loss:

## Re-train with the best model

In [ ]:
lr, num_layers, hidden_size, batch_size, optimizer_name, activation = best_params
size_input = X_train.shape[1]
size_output = 2


if num_layers == 2:
    best_model = MLP(size_input, hidden_size, hidden_size, 0, size_output, activation=activation)
elif num_layers == 3:
    best_model = MLP(size_input, hidden_size, hidden_size, hidden_size, size_output, activation=activation)
else:
    raise ValueError("Invalid number of hidden layers")

# Define the optimizer
if optimizer_name == 'Adam':
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
elif optimizer_name == 'SGD':
    optimizer = tf.keras.optimizers.SGD(learning_rate=lr)
elif optimizer_name == 'RMSProp':
    optimizer = tf.keras.optimizers.RMSprop(learning_rate=lr)
else:
    raise ValueError(f"Invalid optimizer: {optimizer_name}")

# re-train the best model on full train + val data
X_train_full = np.concatenate((X_train, X_val), axis=0)
y_train_full = np.concatenate((y_train, y_val), axis=0)

num_batches = int(np.ceil(X_train_full.shape[0] / batch_size))

for epoch in range(epochs):
    # Shuffle training data at the start of each epoch.
    indices = np.arange(X_train_full.shape[0])
    np.random.shuffle(indices)
    X_train_full = X_train_full[indices]
    y_train_full = y_train_full[indices]

    epoch_loss = 0
    for i in range(num_batches):
        start = i * batch_size
        end = min((i+1) * batch_size, X_train_full.shape[0])
        X_batch = X_train_full[start:end]
        y_batch = y_train_full[start:end]

        # Compute gradients and update weights.
        predictions = best_model.forward(X_batch)
        loss_value = best_model.loss(predictions, y_batch)
        grads = best_model.backward(X_batch, y_batch)
        optimizer.apply_gradients(zip(grads, best_model.variables))
        epoch_loss += loss_value.numpy() * (end - start)

    epoch_loss /= X_train_full.shape[0]

    print(f"Epoch {epoch+1:02d} | Training Loss: {epoch_loss:.4f}")



Epoch 01 | Training Loss: 0.4708
Epoch 02 | Training Loss: 0.0400
Epoch 03 | Training Loss: 0.0063
Epoch 04 | Training Loss: 0.0018
Epoch 05 | Training Loss: 0.0008


## Evaluate the Best Model using Batch Testing 

In [ ]:
# Split the test set into 10 batches
num_test_batches = 10
test_batch_size = X_test.shape[0] // num_test_batches

test_accuracies = []
test_precisions = []
test_recalls = []
test_losses = []

for i in range(num_test_batches):
    start = i * test_batch_size
    end = (i + 1) * test_batch_size if i < num_test_batches - 1 else X_test.shape[0]
    
    X_test_batch = X_test[start:end]
    y_test_batch = y_test[start:end]
    
    test_logits = best_model.forward(X_test_batch)
    test_loss = best_model.loss(test_logits, y_test_batch).numpy()
    test_preds = np.argmax(test_logits.numpy(), axis=1)
    true_test = np.argmax(y_test_batch, axis=1)
    
    test_accuracy = np.mean(test_preds == true_test)
    test_precision = precision_score(true_test, test_preds)
    test_recall = recall_score(true_test, test_preds)
    
    test_accuracies.append(test_accuracy)
    test_precisions.append(test_precision)
    test_recalls.append(test_recall)
    test_losses.append(test_loss)

# Calculate average metrics
avg_test_accuracy = np.mean(test_accuracies)
avg_test_precision = np.mean(test_precisions)
avg_test_recall = np.mean(test_recalls)
avg_test_loss = np.mean(test_losses)

print(f"Average Test Loss: {avg_test_loss:.4f}")
print(f"Average Test Accuracy: {avg_test_accuracy:.4f}")
print(f"Average Test Precision: {avg_test_precision:.4f}")
print(f"Average Test Recall: {avg_test_recall:.4f}")

# Print individual batch results
print("\nIndividual Batch Results:")
for i, (acc, prec, rec) in enumerate(zip(test_accuracies, test_precisions, test_recalls)):
    print(f"Batch {i+1}: Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")





Average Test Loss: 0.4837
Average Test Accuracy: 0.8723
Average Test Precision: 0.8742
Average Test Recall: 0.8697

Individual Batch Results:
Batch 1: Accuracy: 0.8804, Precision: 0.8838, Recall: 0.8803
Batch 2: Accuracy: 0.8664, Precision: 0.8619, Recall: 0.8661
Batch 3: Accuracy: 0.8704, Precision: 0.8667, Recall: 0.8609
Batch 4: Accuracy: 0.8760, Precision: 0.8828, Recall: 0.8673
Batch 5: Accuracy: 0.8596, Precision: 0.8644, Recall: 0.8583
Batch 6: Accuracy: 0.8764, Precision: 0.8796, Recall: 0.8747
Batch 7: Accuracy: 0.8704, Precision: 0.8764, Recall: 0.8681
Batch 8: Accuracy: 0.8740, Precision: 0.8688, Recall: 0.8793
Batch 9: Accuracy: 0.8724, Precision: 0.8713, Recall: 0.8775
Batch 10: Accuracy: 0.8768, Precision: 0.8859, Recall: 0.8646


## Implement the Random MLP with the best parmeters

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, accuracy_score
import itertools

tf.random.set_seed(000)
np.random.seed(000)


import tensorflow as tf
print(tf.test.gpu_device_name())
print('GPU:', tf.config.list_physical_devices(device_type='GPU'))
print(tf.test.is_gpu_available())


class MLP_Random(object):
    def __init__(self, size_input, size_hidden1, size_hidden2, size_hidden3, size_output,
                 activation='relu', device=None):  # Added activation
        """
        size_input: int, size of input layer
        size_hidden1: int, size of the 1st hidden layer
        size_hidden2: int, size of the 2nd hidden layer
        size_hidden3: int, size of the 3rd hidden layer (not used in compute_output here)
        size_output: int, size of output layer
        activation: str, activation function ('relu', 'tanh', 'leaky_relu')
        device: str or None, either 'cpu' or 'gpu' or None.
        """
        self.size_input = size_input
        self.size_hidden1 = size_hidden1
        self.size_hidden2 = size_hidden2
        self.size_hidden3 = size_hidden3
        self.size_output = size_output
        self.activation = activation  # Store activation function
        self.device = device

        # Initialize weights and biases for first hidden layer
        self.W1 = tf.Variable(tf.random.normal([self.size_input, self.size_hidden1], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([1, self.size_hidden1]))

        # Initialize weights and biases for second hidden layer
        self.W2 = tf.Variable(tf.random.normal([self.size_hidden1, self.size_hidden2], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([1, self.size_hidden2]))

        # Initialize weights and biases for third hidden layer (if needed)
        self.W3 = tf.Variable(tf.random.normal([self.size_hidden2, self.size_output], stddev=0.1))
        self.b3 = tf.Variable(tf.zeros([1, self.size_output]))

        # List of variables to update during backpropagation
        self.variables = [self.W3, self.b3]

    def forward(self, X):
        """
        Forward pass.
        X: Tensor, inputs.
        """
        if self.device is not None:
            with tf.device('gpu:0' if self.device == 'gpu' else 'cpu'):
                self.y = self.compute_output(X)
        else:
            self.y = self.compute_output(X)
        return self.y

    def loss(self, y_pred, y_true):
        """
        Computes the loss between predicted and true outputs.
        y_pred: Tensor of shape (batch_size, size_output)
        y_true: Tensor of shape (batch_size, size_output)
        """
        y_true_tf = tf.cast(y_true, dtype=tf.float32)
        y_pred_tf = tf.cast(y_pred, dtype=tf.float32)
        cce = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
        loss_x = cce(y_true_tf, y_pred_tf) and Test
        with tf.GradientTape() as tape:
            predicted = self.forward(X_train)
            current_loss = self.loss(predicted, y_train)
        grads = tape.gradient(current_loss, self.variables)
        return grads

    def compute_output(self, X):
        """
        Custom method to compute the output tensor during the forward pass.
        """
        # Cast X to float32
        X_tf = tf.cast(X, dtype=tf.float32)
        
        # First hidden layer
        h1 = tf.matmul(X_tf, self.W1) + self.b1
        z1 = self.apply_activation(h1)

        # Second hidden layer
        h2 = tf.matmul(z1, self.W2) + self.b2
        z2 = self.apply_activation(h2)

        # Output layer (logits)
        output = tf.matmul(z2, self.W3) + self.b3
        return output

    def apply_activation(self, tensor):

        """Applies the specified activation function."""
        if self.activation == 'relu':
            return tf.nn.relu(tensor)
        elif self.activation == 'tanh':
            return tf.nn.tanh(tensor)
        else:
            raise ValueError(f"Invalid activation function: {self.activation}")


/device:GPU:0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU')]
True


I0000 00:00:1740038289.141700  280541 gpu_device.cc:2022] Created device /device:GPU:0 with 22991 MB memory:  -> device: 0, name: NVIDIA TITAN RTX, pci bus id: 0000:3b:00.0, compute capability: 7.5
I0000 00:00:1740038289.141961  280541 gpu_device.cc:2022] Created device /device:GPU:1 with 22991 MB memory:  -> device: 1, name: NVIDIA TITAN RTX, pci bus id: 0000:86:00.0, compute capability: 7.5
I0000 00:00:1740038289.142184  280541 gpu_device.cc:2022] Created device /device:GPU:2 with 10699 MB memory:  -> device: 2, name: NVIDIA TITAN Xp, pci bus id: 0000:af:00.0, compute capability: 6.1
I0000 00:00:1740038289.146413  280541 gpu_device.cc:2022] Created device /device:GPU:0 with 22991 MB memory:  -> device: 0, name: NVIDIA TITAN RTX, pci bus id: 0000:3b:00.0, compute capability: 7.5
I0000 00:00:1740038289.146669  280541 gpu_device.cc:2022] Created device /device:GPU:1 with 22991 MB memory:  -> device: 1, name: NVIDIA TITAN RTX, pci bus id: 0000:86:00.0, compute capability: 7.5
I0000 00:00

## Loading and preprocess the dataset for Random MLP

In [117]:
# -------------------------------
# Load and Prepare the IMDB Dataset
# -------------------------------
print("Loading IMDB dataset...")
# Load the IMDB reviews dataset with the 'as_supervised' flag so that we get (text, label) pairs.
(ds_train, ds_test), ds_info = tfds.load('imdb_reviews',
                                           split=['train', 'test'],
                                           as_supervised=True,
                                           with_info=True)

# Convert training dataset to lists.
train_texts = []
train_labels = []
for text, label in tfds.as_numpy(ds_train):
    # Decode byte strings to utf-8 strings.
    train_texts.append(text.decode('utf-8'))
    train_labels.append(label)
train_labels = np.array(train_labels)

# Create a validation set from the training data (20% for validation).
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=42)

# Convert test dataset to lists.
test_texts = []
test_labels = []
for text, label in tfds.as_numpy(ds_test):
    test_texts.append(text.decode('utf-8'))
    test_labels.append(label)
test_labels = np.array(test_labels)

print(f"Train samples: {len(train_texts)}, Validation samples: {len(val_texts)}, Test samples: {len(test_texts)}")

# -------------------------------
# Preprocessing: Tokenization and Vectorization
# -------------------------------
# Build the character-level tokenizer on the training texts.
tokenizer = word_level_tokenizer(train_texts)
print("Tokenizer vocabulary size:", len(tokenizer.word_index) + 1)

# Convert texts to bag-of-characters representation.
X_train = texts_to_bow(tokenizer, train_texts)
X_val   = texts_to_bow(tokenizer, val_texts)
X_test  = texts_to_bow(tokenizer, test_texts)

# Convert labels to one-hot encoding.
y_train = one_hot_encode(train_labels)
y_val   = one_hot_encode(val_labels)
y_test  = one_hot_encode(test_labels)



Loading IMDB dataset...
Train samples: 20000, Validation samples: 5000, Test samples: 25000
Tokenizer vocabulary size: 80169


## Train for Random MLP

In [118]:


lr, num_layers, hidden_size, batch_size, optimizer_name, activation = best_params
size_input = X_train.shape[1]
size_output = 2

if num_layers == 2:
    best_model = MLP_Random(size_input, hidden_size, hidden_size, 0, size_output, activation=activation)
elif num_layers == 3:
    best_model = MLP_Random(size_input, hidden_size, hidden_size, hidden_size, size_output, activation=activation)
else:
    raise ValueError("Invalid number of hidden layers")

# Define the optimizer
if optimizer_name == 'Adam':
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
elif optimizer_name == 'SGD':
    optimizer = tf.keras.optimizers.SGD(learning_rate=lr)

else:
    raise ValueError(f"Invalid optimizer: {optimizer_name}")

# re-train the best model on full train + val data
X_train_full = np.concatenate((X_train, X_val), axis=0)
y_train_full = np.concatenate((y_train, y_val), axis=0)

num_batches = int(np.ceil(X_train_full.shape[0] / batch_size))

for epoch in range(epochs):
    # Shuffle training data at the start of each epoch.
    indices = np.arange(X_train_full.shape[0])
    np.random.shuffle(indices)
    X_train_full = X_train_full[indices]
    y_train_full = y_train_full[indices]

    epoch_loss = 0
    for i in range(num_batches):
        start = i * batch_size
        end = min((i+1) * batch_size, X_train_full.shape[0])
        X_batch = X_train_full[start:end]
        y_batch = y_train_full[start:end]

        # Compute gradients and update weights.
        predictions = best_model.forward(X_batch)
        loss_value = best_model.loss(predictions, y_batch)
        grads = best_model.backward(X_batch, y_batch)
        optimizer.apply_gradients(zip(grads, best_model.variables))
        epoch_loss += loss_value.numpy() * (end - start)

    epoch_loss /= X_train_full.shape[0]

    print(f"Epoch {epoch+1:02d} | Training Loss: {epoch_loss:.4f}")

Epoch 01 | Training Loss: 0.9545
Epoch 02 | Training Loss: 0.6890
Epoch 03 | Training Loss: 0.6265
Epoch 04 | Training Loss: 0.6013
Epoch 05 | Training Loss: 0.5937


## Test for Random MLP

In [119]:
# Split the test set into 10 batches
num_test_batches = 10
test_batch_size = X_test.shape[0] // num_test_batches

test_accuracies = []
test_precisions = []
test_recalls = []
test_losses = []

for i in range(num_test_batches):
    start = i * test_batch_size
    end = (i + 1) * test_batch_size if i < num_test_batches - 1 else X_test.shape[0]
    
    X_test_batch = X_test[start:end]
    y_test_batch = y_test[start:end]
    
    test_logits = best_model.forward(X_test_batch)
    test_loss = best_model.loss(test_logits, y_test_batch).numpy()
    test_preds = np.argmax(test_logits.numpy(), axis=1)
    true_test = np.argmax(y_test_batch, axis=1)
    
    test_accuracy = np.mean(test_preds == true_test)
    test_precision = precision_score(true_test, test_preds)
    test_recall = recall_score(true_test, test_preds)
    
    test_accuracies.append(test_accuracy)
    test_precisions.append(test_precision)
    test_recalls.append(test_recall)
    test_losses.append(test_loss)

# Calculate average metrics
avg_test_accuracy = np.mean(test_accuracies)
avg_test_precision = np.mean(test_precisions)
avg_test_recall = np.mean(test_recalls)
avg_test_loss = np.mean(test_losses)

print(f"Average Test Loss: {avg_test_loss:.4f}")
print(f"Average Test Accuracy: {avg_test_accuracy:.4f}")
print(f"Average Test Precision: {avg_test_precision:.4f}")
print(f"Average Test Recall: {avg_test_recall:.4f}")

# Print individual batch results
print("\nIndividual Batch Results:")
for i, (acc, prec, rec) in enumerate(zip(test_accuracies, test_precisions, test_recalls)):
    print(f"Batch {i+1}: Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")


Average Test Loss: 0.5988
Average Test Accuracy: 0.6861
Average Test Precision: 0.6748
Average Test Recall: 0.7185

Individual Batch Results:
Batch 1: Accuracy: 0.6904, Precision: 0.6837, Recall: 0.7268
Batch 2: Accuracy: 0.6852, Precision: 0.6657, Recall: 0.7184
Batch 3: Accuracy: 0.6876, Precision: 0.6558, Recall: 0.7267
Batch 4: Accuracy: 0.6952, Precision: 0.6840, Recall: 0.7266
Batch 5: Accuracy: 0.6896, Precision: 0.6838, Recall: 0.7236
Batch 6: Accuracy: 0.6832, Precision: 0.6783, Recall: 0.7074
Batch 7: Accuracy: 0.6804, Precision: 0.6792, Recall: 0.7064
Batch 8: Accuracy: 0.6904, Precision: 0.6772, Recall: 0.7208
Batch 9: Accuracy: 0.6808, Precision: 0.6736, Recall: 0.7162
Batch 10: Accuracy: 0.6784, Precision: 0.6664, Recall: 0.7123


# Part 3: MLP with FA

In [7]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score

tf.random.set_seed(12)
np.random.seed(12)


import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score

class MLP_FA(object):
    def __init__(self, size_input, size_hidden1, size_hidden2, size_hidden3, size_hidden4, size_hidden5, size_hidden6, size_output, device=None):
        """
        size_input: int, size of input layer
        size_hidden1: int, size of the 1st hidden layer
        size_hidden2: int, size of the 2nd hidden layer
        size_hidden3: int, size of the 3rd hidden layer
        size_hidden4: int, size of the 4th hidden layer
        size_hidden5: int, size of the 5th hidden layer
        size_hidden6: int, size of the 6th hidden layer
        size_output: int, size of output layer
        device: str or None, either 'cpu' or 'gpu' or None.
        """
        self.size_input = size_input
        self.size_hidden1 = size_hidden1
        self.size_hidden2 = size_hidden2
        self.size_hidden3 = size_hidden3
        self.size_hidden4 = size_hidden4
        self.size_hidden5 = size_hidden5
        self.size_hidden6 = size_hidden6
        self.size_output = size_output
        self.device = device

        # Initialize weights and biases for first hidden layer
        self.W1 = tf.Variable(tf.random.normal([self.size_input, self.size_hidden1], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([1, self.size_hidden1]))

        # Initialize weights and biases for second hidden layer
        self.W2 = tf.Variable(tf.random.normal([self.size_hidden1, self.size_hidden2], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([1, self.size_hidden2]))

        # Initialize weights and biases for third hidden layer
        self.W3 = tf.Variable(tf.random.normal([self.size_hidden2, self.size_hidden3], stddev=0.1))
        self.b3 = tf.Variable(tf.zeros([1, self.size_hidden3]))

        # Initialize weights and biases for fourth hidden layer
        self.W4 = tf.Variable(tf.random.normal([self.size_hidden3, self.size_hidden4], stddev=0.1))
        self.b4 = tf.Variable(tf.zeros([1, self.size_hidden4]))

        # Initialize weights and biases for fifth hidden layer
        self.W5 = tf.Variable(tf.random.normal([self.size_hidden4, self.size_hidden5], stddev=0.1))
        self.b5 = tf.Variable(tf.zeros([1, self.size_hidden5]))

        # Initialize weights and biases for sixth hidden layer
        self.W6 = tf.Variable(tf.random.normal([self.size_hidden5, self.size_hidden6], stddev=0.1))
        self.b6 = tf.Variable(tf.zeros([1, self.size_hidden6]))

        # Initialize weights and biases for output layer
        self.W_out = tf.Variable(tf.random.normal([self.size_hidden6, self.size_output], stddev=0.1))
        self.b_out = tf.Variable(tf.zeros([1, self.size_output]))

        # Create fixed random feedback matrices for feedback alignment:
        self.B_out = tf.Variable(tf.random.normal([self.size_output, self.size_hidden6]), trainable=False)
        self.B6 = tf.Variable(tf.random.normal([self.size_hidden6, self.size_hidden5]), trainable=False)
        self.B5 = tf.Variable(tf.random.normal([self.size_hidden5, self.size_hidden4]), trainable=False)
        self.B4 = tf.Variable(tf.random.normal([self.size_hidden4, self.size_hidden3]), trainable=False)
        self.B3 = tf.Variable(tf.random.normal([self.size_hidden3, self.size_hidden2]), trainable=False)
        self.B2 = tf.Variable(tf.random.normal([self.size_hidden2, self.size_hidden1]), trainable=False)

        # Define variables to be updated during training
        self.variables = [self.W1, self.W2, self.W3, self.W4, self.W5, self.W6, self.W_out,
                          self.b1, self.b2, self.b3, self.b4, self.b5, self.b6, self.b_out]

    def forward(self, X):
        """
        Forward pass.
        X: Tensor, inputs.
        """
        if self.device is not None:
            with tf.device('gpu:0' if self.device == 'gpu' else 'cpu'):
                self.y = self.compute_output(X)
        else:
            self.y = self.compute_output(X)
        return self.y

    def loss(self, y_pred, y_true):
        """
        Computes the loss between predicted and true outputs.
        y_pred - Tensor of shape (batch_size, size_output)
        y_true - Tensor of shape (batch_size, size_output)
        """
        y_true_tf = tf.cast(y_true, dtype=tf.float32)
        y_pred_tf = tf.cast(y_pred, dtype=tf.float32)
        cce = tf.keras.losses.CategoricalCrossentropy(from_logits=True)
        loss_x = cce(y_true_tf, y_pred_tf)
        return loss_x

    def backward(self, X_train, y_train):
        """
        Backward pass using feedback alignment.
        Computes gradients manually using fixed random feedback matrices.
        X_train: Input data (numpy array)
        y_train: One-hot encoded labels (numpy array)
        Returns: List of gradients corresponding to [dW1, dW2, dW3, dW4, dW5, dW6, dW_out, db1, db2, db3, db4, db5, db6, db_out]
        """
        # Cast input to float32 tensor
        X_tf = tf.cast(X_train, tf.float32)
        batch_size = tf.cast(tf.shape(X_tf)[0], tf.float32)

        # --- Forward Pass ---
        h1 = tf.matmul(X_tf, self.W1) + self.b1
        a1 = tf.nn.relu(h1)

        h2 = tf.matmul(a1, self.W2) + self.b2
        a2 = tf.nn.relu(h2)

        h3 = tf.matmul(a2, self.W3) + self.b3
        a3 = tf.nn.relu(h3)

        h4 = tf.matmul(a3, self.W4) + self.b4
        a4 = tf.nn.relu(h4)

        h5 = tf.matmul(a4, self.W5) + self.b5
        a5 = tf.nn.relu(h5)

        h6 = tf.matmul(a5, self.W6) + self.b6
        a6 = tf.nn.relu(h6)

        logits = tf.matmul(a6, self.W_out) + self.b_out
        y_pred = tf.nn.softmax(logits)

        # --- Compute Output Error ---
        delta_out = y_pred - tf.cast(y_train, tf.float32)

        dW_out = tf.matmul(tf.transpose(a6), delta_out) / batch_size
        db_out = tf.reduce_mean(delta_out, axis=0, keepdims=True)

        # --- Feedback Alignment ---
        relu_grad_h6 = tf.cast(h6 > 0, tf.float32)
        delta6 = tf.matmul(delta_out, self.B_out) * relu_grad_h6
        dW6 = tf.matmul(tf.transpose(a5), delta6) / batch_size
        db6 = tf.reduce_mean(delta6, axis=0, keepdims=True)

        relu_grad_h5 = tf.cast(h5 > 0, tf.float32)
        delta5 = tf.matmul(delta6, self.B6) * relu_grad_h5
        dW5 = tf.matmul(tf.transpose(a4), delta5) / batch_size
        db5 = tf.reduce_mean(delta5, axis=0, keepdims=True)

        relu_grad_h4 = tf.cast(h4 > 0, tf.float32)
        delta4 = tf.matmul(delta5, self.B5) * relu_grad_h4
        dW4 = tf.matmul(tf.transpose(a3), delta4) / batch_size
        db4 = tf.reduce_mean(delta4, axis=0, keepdims=True)

        relu_grad_h3 = tf.cast(h3 > 0, tf.float32)
        delta3 = tf.matmul(delta4, self.B4) * relu_grad_h3
        dW3 = tf.matmul(tf.transpose(a2), delta3) / batch_size
        db3 = tf.reduce_mean(delta3, axis=0, keepdims=True)

        relu_grad_h2 = tf.cast(h2 > 0, tf.float32)
        delta2 = tf.matmul(delta3, self.B3) * relu_grad_h2
        dW2 = tf.matmul(tf.transpose(a1), delta2) / batch_size
        db2 = tf.reduce_mean(delta2, axis=0, keepdims=True)

        relu_grad_h1 = tf.cast(h1 > 0, tf.float32)
        delta1 = tf.matmul(delta2, self.B2) * relu_grad_h1
        dW1 = tf.matmul(tf.transpose(X_tf), delta1) / batch_size
        db1 = tf.reduce_mean(delta1, axis=0, keepdims=True)

        return [dW1, dW2, dW3, dW4, dW5, dW6, dW_out, db1, db2, db3, db4, db5, db6, db_out]

    def compute_output(self, X):
        """
        Custom method to obtain output tensor during the forward pass.
        """
        X_tf = tf.cast(X, dtype=tf.float32)

        h1 = tf.matmul(X_tf, self.W1) + self.b1
        z1 = tf.nn.relu(h1)

        h2 = tf.matmul(z1, self.W2) + self.b2
        z2 = tf.nn.relu(h2)

        h3 = tf.matmul(z2, self.W3) + self.b3
        z3 = tf.nn.relu(h3)

        h4 = tf.matmul(z3, self.W4) + self.b4
        z4 = tf.nn.relu(h4)

        h5 = tf.matmul(z4, self.W5) + self.b5
        z5 = tf.nn.relu(h5)

        h6 = tf.matmul(z5, self.W6) + self.b6
        z6 = tf.nn.relu(h6)

        output = tf.matmul(z6, self.W_out) + self.b_out
        return output

# -------------------------------
# Load and Prepare the IMDB Dataset
# -------------------------------
print("Loading IMDB dataset...")
# Load the IMDB reviews dataset with the 'as_supervised' flag so that we get (text, label) pairs.
(ds_train, ds_test), ds_info = tfds.load('imdb_reviews',
                                           split=['train', 'test'],
                                           as_supervised=True,
                                           with_info=True)

# Convert training dataset to lists.
train_texts = []
train_labels = []
for text, label in tfds.as_numpy(ds_train):
    # Decode byte strings to utf-8 strings.
    train_texts.append(text.decode('utf-8'))
    train_labels.append(label)
train_labels = np.array(train_labels)

# Create a validation set from the training data (20% for validation).
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.2, random_state=42)

# Convert test dataset to lists.
test_texts = []
test_labels = []
for text, label in tfds.as_numpy(ds_test):
    test_texts.append(text.decode('utf-8'))
    test_labels.append(label)
test_labels = np.array(test_labels)

print(f"Train samples: {len(train_texts)}, Validation samples: {len(val_texts)}, Test samples: {len(test_texts)}")

# -------------------------------
# Preprocessing: Tokenization and Vectorization
# -------------------------------
# Build the character-level tokenizer on the training texts.
tokenizer = word_level_tokenizer(train_texts)
print("Tokenizer vocabulary size:", len(tokenizer.word_index) + 1)

# Convert texts to bag-of-characters representation.
X_train = texts_to_bow(tokenizer, train_texts)
X_val   = texts_to_bow(tokenizer, val_texts)
X_test  = texts_to_bow(tokenizer, test_texts)

# Convert labels to one-hot encoding.
y_train = one_hot_encode(train_labels)
y_val   = one_hot_encode(val_labels)
y_test  = one_hot_encode(test_labels)

# -------------------------------
# Model Setup
# -------------------------------
# The input size is determined by the dimension of the bag-of-characters vector.
size_input = X_train.shape[1]
# Set hidden layer sizes as desired.
size_hidden1 = 512
size_hidden2 = 256
size_hidden3 = 128
size_hidden4 = 64
size_hidden5 = 32
size_hidden6 = 16
size_output  = 2

# Instantiate the MLP model.
model = MLP_FA(size_input, size_hidden1, size_hidden2, size_hidden3, size_hidden4, size_hidden5, size_hidden6, size_output, device=None)

# Define the optimizer.
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# -------------------------------
# Training Parameters and Loop
# -------------------------------
batch_size = 128
epochs = 10
num_batches = int(np.ceil(X_train.shape[0] / batch_size))

print("\nStarting training...\n")
for epoch in range(epochs):
    # Shuffle training data at the start of each epoch.
    indices = np.arange(X_train.shape[0])
    np.random.shuffle(indices)
    X_train = X_train[indices]
    y_train = y_train[indices]

    epoch_loss = 0
    for i in range(num_batches):
        start = i * batch_size
        end = min((i+1) * batch_size, X_train.shape[0])
        X_batch = X_train[start:end]
        y_batch = y_train[start:end]

        # Compute gradients and update weights.
        # with tf.GradientTape() as tape:
        #     predictions = model.forward(X_batch)
        #     loss_value = model.loss(predictions, y_batch)
        # grads = tape.gradient(loss_value, model.variables)
        predictions = model.forward(X_batch)
        loss_value = model.loss(predictions, y_batch)
        grads = model.backward(X_batch, y_batch)
        optimizer.apply_gradients(zip(grads, model.variables))
        epoch_loss += loss_value.numpy() * (end - start)

    epoch_loss /= X_train.shape[0]

    # Evaluate on validation set.
    val_logits = model.forward(X_val)
    val_loss = model.loss(val_logits, y_val).numpy()
    val_preds = np.argmax(val_logits.numpy(), axis=1)
    true_val = np.argmax(y_val, axis=1)
    accuracy = np.mean(val_preds == true_val)
    precision = precision_score(true_val, val_preds)
    recall = recall_score(true_val, val_preds)

    print(f"Epoch {epoch+1:02d} | Training Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

# -------------------------------
# Final Evaluation on Test Set
# -------------------------------

# Split the test set into 10 batches
num_test_batches = 10
test_batch_size = X_test.shape[0] // num_test_batches

test_accuracies = []
test_precisions = []
test_recalls = []
test_losses = []

for i in range(num_test_batches):
    start = i * test_batch_size
    end = (i + 1) * test_batch_size if i < num_test_batches - 1 else X_test.shape[0]
    
    X_test_batch = X_test[start:end]
    y_test_batch = y_test[start:end]
    
    test_logits = model.forward(X_test_batch)
    test_loss = model.loss(test_logits, y_test_batch).numpy()
    test_preds = np.argmax(test_logits.numpy(), axis=1)
    true_test = np.argmax(y_test_batch, axis=1)
    
    test_accuracy = np.mean(test_preds == true_test)
    test_precision = precision_score(true_test, test_preds)
    test_recall = recall_score(true_test, test_preds)
    
    test_accuracies.append(test_accuracy)
    test_precisions.append(test_precision)
    test_recalls.append(test_recall)
    test_losses.append(test_loss)

# Calculate average metrics
avg_test_accuracy = np.mean(test_accuracies)
avg_test_precision = np.mean(test_precisions)
avg_test_recall = np.mean(test_recalls)
avg_test_loss = np.mean(test_losses)

print(f"Average Test Loss: {avg_test_loss:.4f}")
print(f"Average Test Accuracy: {avg_test_accuracy:.4f}")
print(f"Average Test Precision: {avg_test_precision:.4f}")
print(f"Average Test Recall: {avg_test_recall:.4f}")

# Print individual batch results
print("\nIndividual Batch Results:")
for i, (acc, prec, rec) in enumerate(zip(test_accuracies, test_precisions, test_recalls)):
    print(f"Batch {i+1}: Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}")


Loading IMDB dataset...
Train samples: 20000, Validation samples: 5000, Test samples: 25000
Tokenizer vocabulary size: 80169

Starting training...

Epoch 01 | Training Loss: 0.4536 | Val Loss: 0.3031 | Accuracy: 0.8772 | Precision: 0.8369 | Recall: 0.9274
Epoch 02 | Training Loss: 0.1490 | Val Loss: 0.2993 | Accuracy: 0.8888 | Precision: 0.8812 | Recall: 0.8907
Epoch 03 | Training Loss: 0.0443 | Val Loss: 0.4795 | Accuracy: 0.8838 | Precision: 0.8573 | Recall: 0.9121
Epoch 04 | Training Loss: 0.0111 | Val Loss: 0.5981 | Accuracy: 0.8810 | Precision: 0.8593 | Recall: 0.9022
Epoch 05 | Training Loss: 0.0023 | Val Loss: 0.6879 | Accuracy: 0.8834 | Precision: 0.8768 | Recall: 0.8837
Epoch 06 | Training Loss: 0.0006 | Val Loss: 0.7835 | Accuracy: 0.8820 | Precision: 0.8622 | Recall: 0.9006
Epoch 07 | Training Loss: 0.0002 | Val Loss: 0.8201 | Accuracy: 0.8826 | Precision: 0.8705 | Recall: 0.8903
Epoch 08 | Training Loss: 0.0001 | Val Loss: 0.8625 | Accuracy: 0.8824 | Precision: 0.8705 | Rec